In [2]:
import json

# Et dictionary er et key value objeckt

personlige_oplysninger = { 'højde': 196, 'alder' : 55}

print (personlige_oplysninger)
print (type(personlige_oplysninger))

{'højde': 196, 'alder': 55}
<class 'dict'>


In [3]:
json_streng = json.dumps(personlige_oplysninger)

print (json_streng)
print (type(json_streng))

{"h\u00f8jde": 196, "alder": 55}
<class 'str'>


In [4]:
donald_dict = {'Fox news' : True, 'All others' : False }

print (donald_dict)
print (type(donald_dict))

{'Fox news': True, 'All others': False}
<class 'dict'>


In [5]:
donald_json = json.dumps(donald_dict)

print (donald_json)
print (type(donald_json))

{"Fox news": true, "All others": false}
<class 'str'>


In [8]:
from encodings import utf_8

with open("donald-trusted-news-index.json", mode = "w", encoding="utf_8") as donald_fil:
    json.dump(donald_dict, donald_fil)

In [13]:
import json
from datetime import datetime
from typing import List
from dataclasses import dataclass

In [ ]:

@dataclass
class ElprisDC:
    hour_utc : datetime
    hour_dk : datetime
    price_area : str
    spot_price_dkk : float
    spot_price_eur : float

    ''' Class method er kun et eksempel stjålet nedenfra vi skal have en form for fra dictionary til dataclass
        for at kunne serialisere og deserialisere og selv om vi så bruger en dataclass så får vi ekstra meget
        boilerplate kode der gør det uoverskuelig i en i forvejen uoverskuelig python syntax.

        Og der ser vi i næste uge i tomat source 35 hvordan det kan gøres endnu simplere med "Pydantic"
    '''
    @classmethod
    def fra_dict(cls, data: dict):
        # Konverterer dato-strenge til datetime-objekter
        utc_tid = datetime.fromisoformat(data['HourUTC'])
        dk_tid = datetime.fromisoformat(data['HourDK'])
        
        return cls(
            HourUTC=utc_tid,
            HourDK=dk_tid,
            PriceArea=data['PriceArea'],
            SpotPriceDKK=data['SpotPriceDKK'],
            SpotPriceEUR=data['SpotPriceEUR']
        )
 


class ElprisRecord:
    """En klasse der repræsenterer et enkelt prispunkt."""
    def __init__(self, HourUTC: datetime, HourDK: datetime, PriceArea: str, SpotPriceDKK: float, SpotPriceEUR: float):
        self.HourUTC = HourUTC
        self.HourDK = HourDK
        self.PriceArea = PriceArea
        self.SpotPriceDKK = SpotPriceDKK
        self.SpotPriceEUR = SpotPriceEUR

    @classmethod
    def fra_dict(cls, data: dict) -> ElprisRecord:
        # Konverterer dato-strenge til datetime-objekter
        utc_tid = datetime.fromisoformat(data['HourUTC'])
        dk_tid = datetime.fromisoformat(data['HourDK'])
        
        # Konstruer et objekt af typen ElprisRecord.
        # cls er et alias for typen. Altså ElprisRecord
        return cls(
            HourUTC=utc_tid,
            HourDK=dk_tid,
            PriceArea=data['PriceArea'],
            SpotPriceDKK=data['SpotPriceDKK'],
            SpotPriceEUR=data['SpotPriceEUR']
        )
    

class Elpriser:
    """Hovedklassen der repræsenterer hele JSON-filens indhold."""
    def __init__(self, total: int, filters: str, sort: str, dataset: str, records: List[ElprisRecord]):
        self.total = total
        self.filters = filters
        self.sort = sort
        self.dataset = dataset
        self.records = records

    @classmethod
    def fra_json(cls, json_data: dict):
        # Omdanner 'records' til en liste af ElprisRecord-objekter med type-hints
        records = [ElprisRecord.fra_dict(record_dict) for record_dict in json_data['records']]
        return cls(
            total=json_data['total'],
            filters=json_data['filters'],
            sort=json_data['sort'],
            dataset=json_data['dataset'],
            records=records
        )

In [3]:
json_indhold = {}

with open("elpris-raw.json", mode='r') as json_fil:
    json_indhold = json.load(json_fil)

print (json_indhold)
print (type(json_indhold))

{'total': 24, 'filters': '{"PriceArea":"DK1"}', 'sort': 'HourDK', 'dataset': 'Elspotprices', 'records': [{'HourUTC': '2025-08-18T22:00:00', 'HourDK': '2025-08-19T00:00:00', 'PriceArea': 'DK1', 'SpotPriceDKK': 753.928554, 'SpotPriceEUR': 101.010002}, {'HourUTC': '2025-08-18T23:00:00', 'HourDK': '2025-08-19T01:00:00', 'PriceArea': 'DK1', 'SpotPriceDKK': 707.726998, 'SpotPriceEUR': 94.82}, {'HourUTC': '2025-08-19T00:00:00', 'HourDK': '2025-08-19T02:00:00', 'PriceArea': 'DK1', 'SpotPriceDKK': 658.315958, 'SpotPriceEUR': 88.199997}, {'HourUTC': '2025-08-19T01:00:00', 'HourDK': '2025-08-19T03:00:00', 'PriceArea': 'DK1', 'SpotPriceDKK': 644.657065, 'SpotPriceEUR': 86.370003}, {'HourUTC': '2025-08-19T02:00:00', 'HourDK': '2025-08-19T04:00:00', 'PriceArea': 'DK1', 'SpotPriceDKK': 672.422721, 'SpotPriceEUR': 90.089996}, {'HourUTC': '2025-08-19T03:00:00', 'HourDK': '2025-08-19T05:00:00', 'PriceArea': 'DK1', 'SpotPriceDKK': 754.003156, 'SpotPriceEUR': 101.019997}, {'HourUTC': '2025-08-19T04:00:00'

In [15]:
# Deserialiserer dataen
elpriser_objekt = Elpriser.fra_json(json_indhold)

# Nu er 'HourDK' et rigtigt datetime-objekt, hvilket er meget nyttigt
første_record = elpriser_objekt.records[0]
sidste_record = elpriser_objekt.records[23]

print('*****')
print(første_record)
print('*****')


print(f"Type af HourDK: {type(første_record.HourDK)}")
print(f"HourDK-objekt: {første_record.HourDK}")

# Nu kan du nemt formatere datoen og tiden
formateret_tid = første_record.HourDK.strftime('%A den %d. %B, kl. %H:%M')
sidste_tid = sidste_record.HourDK
print(f"sidste tid er {sidste_tid}")
print(f"Typen af sidste tid er {type(sidste_tid)}")
print(f"\\nFormateret tid: {formateret_tid}")
print(f"Typen af formateret tid er {type(formateret_tid)}")
print(f"Pris (DKK): {første_record.SpotPriceDKK}")
print(f"Pris (DKK) Sidste record: {sidste_record.SpotPriceDKK}")

*****
*****
Type af HourDK: <class 'datetime.datetime'>
HourDK-objekt: 2025-08-19 00:00:00
sidste tid er 2025-08-19 23:00:00
Typen af sidste tid er <class 'datetime.datetime'>
\nFormateret tid: Tuesday den 19. August, kl. 00:00
Typen af formateret tid er <class 'str'>
Pris (DKK): 753.928554
Pris (DKK) Sidste record: 722.356235


In [ ]:
@dataclass
class ElprisDC:
    hour_utc : datetime
    hour_dk : datetime
    price_area : str
    spot_price_dkk : float
    spot_price_eur : float

